# Category LSTM — CPU evaluation readiness

In Kaggle choose **Accelerator: None**. GPU is not used. Attach the frozen local
bundle containing model.keras, input_scaler.pkl, target_scaler.pkl, metadata.json.
This notebook verifies hashes and runs generated-history inference. It does not
train, download records, or establish external accuracy.

Use a compatible CPU environment: tensorflow2.20.0, keras3.13.2,
scikit-learn1.6.1. The recorded verification environment also used numpy2.0.2,
pandas2.3.3 and joblib1.5.3. Package installation is deliberately separate.

Important finding: sparse synthetic histories produced very large utilities
forecasts despite no utilities history. Preserve those outputs during evaluation;
do not treat artifact loading success as model-quality approval.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
from pathlib import Path
import sys, hashlib, json
SOURCES = {'category_lstm_features.py': '"""Exact trained category-feature equations from the reviewed notebook.\n\nGenerated by build_category_feature_source.py. Input panel is validated and\nbounded by category_lstm_frozen.py; no notebook globals or model loading here.\n"""\nimport numpy as np\nimport pandas as pd\n\nUSER_COL = \'user_id\'\nSPEND_COLS = [f\'spend_{i}\' for i in range(9)]\nCOUNT_COLS = [f\'count_{i}\' for i in range(9)]\n\ndef add_model_features(panel):\n\n    x = (\n        panel\n        .sort_values(\n            [\n                USER_COL,\n                "week"\n            ]\n        )\n        .reset_index(\n            drop=True\n        )\n        .copy()\n    )\n\n\n    # --------------------------------------------------------\n    # Overall weekly behaviour\n    # --------------------------------------------------------\n\n    x[\n        "log_total_spend"\n    ] = np.log1p(\n        x["total_spend"]\n    )\n\n    x[\n        "log_total_count"\n    ] = np.log1p(\n        x["total_count"]\n    )\n\n\n    safe_total = (\n        x["total_spend"]\n        .replace(\n            0,\n            np.nan\n        )\n    )\n\n\n    feature_cols = []\n\n\n    # --------------------------------------------------------\n    # Per-category behaviour\n    # --------------------------------------------------------\n\n    for i, (\n        spend_col,\n        count_col\n    ) in enumerate(\n        zip(\n            SPEND_COLS,\n            COUNT_COLS\n        )\n    ):\n\n        # ----------------------------\n        # Existing current-week data\n        # ----------------------------\n\n        log_spend_col = (\n            f"log_spend_{i}"\n        )\n\n        log_count_col = (\n            f"log_count_{i}"\n        )\n\n        share_col = (\n            f"share_{i}"\n        )\n\n\n        x[\n            log_spend_col\n        ] = np.log1p(\n            x[spend_col]\n        )\n\n        x[\n            log_count_col\n        ] = np.log1p(\n            x[count_col]\n        )\n\n        x[\n            share_col\n        ] = (\n            x[spend_col]\n            / safe_total\n        ).fillna(\n            0.0\n        )\n\n\n        # ----------------------------------------------------\n        # Existing short-term activity history\n        # ----------------------------------------------------\n\n        for window in [\n            4,\n            8,\n            13\n        ]:\n\n            col = (\n                f"prior_active_rate"\n                f"{window}_{i}"\n            )\n\n            x[col] = (\n                x\n                .groupby(\n                    USER_COL\n                )[spend_col]\n                .transform(\n                    lambda s:\n                    (\n                        s.gt(0)\n                        .astype(float)\n                        .shift(1)\n                        .rolling(\n                            window,\n                            min_periods=1\n                        )\n                        .mean()\n                        .fillna(0.0)\n                    )\n                )\n            )\n\n\n        # ----------------------------------------------------\n        # NEW: same period approximately one year earlier\n        # ----------------------------------------------------\n\n        yearly_lag = (\n            x\n            .groupby(\n                USER_COL\n            )[spend_col]\n            .shift(52)\n            .fillna(0.0)\n        )\n\n        yearly_col = (\n            f"log_spend_lag52_{i}"\n        )\n\n        x[\n            yearly_col\n        ] = np.log1p(\n            yearly_lag\n        )\n\n\n        # ----------------------------------------------------\n        # NEW: prior 13-week spending baseline\n        # ----------------------------------------------------\n\n        prior_mean_13 = (\n            x\n            .groupby(\n                USER_COL\n            )[spend_col]\n            .transform(\n                lambda s:\n                (\n                    s.shift(1)\n                    .rolling(\n                        13,\n                        min_periods=4\n                    )\n                    .mean()\n                )\n            )\n            .fillna(0.0)\n        )\n\n        mean13_col = (\n            f"log_prior_mean13_{i}"\n        )\n\n        x[\n            mean13_col\n        ] = np.log1p(\n            prior_mean_13\n        )\n\n\n        feature_cols.extend([\n            log_spend_col,\n            log_count_col,\n            share_col,\n\n            f"prior_active_rate4_{i}",\n            f"prior_active_rate8_{i}",\n            f"prior_active_rate13_{i}",\n\n            yearly_col,\n            mean13_col,\n        ])\n\n\n    # --------------------------------------------------------\n    # Calendar seasonality\n    # --------------------------------------------------------\n\n    week_number = (\n        x["week"]\n        .dt\n        .isocalendar()\n        .week\n        .astype(int)\n    )\n\n    x[\n        "week_sin"\n    ] = np.sin(\n        2\n        * np.pi\n        * week_number\n        / 52.0\n    )\n\n    x[\n        "week_cos"\n    ] = np.cos(\n        2\n        * np.pi\n        * week_number\n        / 52.0\n    )\n\n\n    feature_cols.extend([\n        "log_total_spend",\n        "log_total_count",\n        "week_sin",\n        "week_cos",\n    ])\n\n\n    return (\n        x,\n        feature_cols\n    )\n', 'category_lstm_frozen.py': '"""Frozen 26-week category LSTM adapter for CPU-only evaluation.\n\nNo fit, recalibration or raw-data download. Explicit observed coverage is required.\nArtifact hashes identify the reviewed Drive final bundle, not the earlier\ndevelopment checkpoint. Joblib loading requires explicit trusted-artifact consent.\n"""\nimport os\n# Must be set before importing TensorFlow, including in the generated notebook.\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'-1\'\n\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.exceptions import InconsistentVersionWarning\nfrom sklearn.preprocessing import StandardScaler\n\nfrom category_lstm_features import add_model_features, SPEND_COLS, COUNT_COLS\n\nHASHES = {\n    \'model.keras\': \'d6250e19063756465bfd9b24760ee368c284ab30b5c4db272480c39263f78e1e\',\n    \'input_scaler.pkl\': \'33acd1d92c6ea76275eedfd747b0133fcdef352a2a542242159b0493377417cd\',\n    \'target_scaler.pkl\': \'50e2c03beaa3f665922a3c2ff50ac926c737ab38a1f0d8137b63f4a7cf1cbc0a\',\n    \'metadata.json\': \'8b51ebe16e6c8725c7f1707bd825e7bd183fc3006a6e3edcedc45dc2033a140c\',\n}\nCATEGORIES = [\'Education\', \'Entertainment\', \'Food\', \'Healthcare\', \'Rent\', \'Shopping\',\n              \'Subscriptions\', \'Transport\', \'Utilities\']\nFEATURES = [name for i in range(9) for name in\n            (f\'log_spend_{i}\', f\'log_count_{i}\', f\'share_{i}\', f\'prior_active_rate4_{i}\',\n             f\'prior_active_rate8_{i}\', f\'prior_active_rate13_{i}\', f\'log_spend_lag52_{i}\',\n             f\'log_prior_mean13_{i}\')]+[\'log_total_spend\', \'log_total_count\', \'week_sin\', \'week_cos\']\n\n\ndef version_record():\n    return {name: importlib.metadata.version(name) for name in\n            (\'tensorflow\', \'keras\', \'numpy\', \'pandas\', \'scikit-learn\', \'joblib\')}\n\n\ndef timestamp(value):\n    result = pd.Timestamp(value)\n    if pd.isna(result) or result.tzinfo is None:\n        raise ValueError(\'Timezone-aware timestamp required\')\n    return result.tz_convert(\'Africa/Nairobi\')\n\n\ndef week_boundary(value):\n    result = timestamp(value)\n    if result.dayofweek != 0 or result != result.normalize():\n        raise ValueError(\'Expected Monday midnight in Africa/Nairobi\')\n    return result\n\n\ndef history_panel(records, *, owner, observed_from, target_start, as_of, history_complete):\n    """Construct only completed, explicitly observed weeks before target_start.\n\nHistory must include all available covered transactions. Never use the target\nweek, fill unconfirmed gaps, or silently discard invalid/unmapped expense rows.\n"""\n    if history_complete is not True:\n        raise ValueError(\'Complete recorded-history attestation required\')\n    start, target, now = week_boundary(observed_from), week_boundary(target_start), timestamp(as_of)\n    weeks = (target-start).days//7\n    if weeks < 26 or target > now:\n        raise ValueError(\'At least 26 complete observed weeks before an available target required\')\n    if not isinstance(owner, str) or not owner.strip():\n        raise ValueError(\'Explicit household identity required\')\n    required = {\'user_id\', \'transaction_id\', \'transaction_timestamp\', \'transaction_type\', \'amount\', \'category\'}\n    if not isinstance(records, pd.DataFrame) or not required <= set(records):\n        raise ValueError(\'Missing transaction fields\')\n    data = records.copy()\n    if data[list(required)].isna().any().any():\n        raise ValueError(\'Missing transaction values\')\n    if not data.user_id.astype(str).eq(owner).all():\n        raise ValueError(\'History contains another household\')\n    if data.transaction_id.astype(str).str.strip().eq(\'\').any() or data.transaction_id.duplicated().any():\n        raise ValueError(\'Nonblank unique transaction IDs required\')\n    if not data.transaction_type.isin([\'expense\', \'income\']).all():\n        raise ValueError(\'Unknown transaction type\')\n    data[\'amount\'] = pd.to_numeric(data.amount, errors=\'raise\')\n    if pd.api.types.is_bool_dtype(data.amount) or not np.isfinite(data.amount).all() or data.amount.lt(0).any():\n        raise ValueError(\'Finite nonnegative KES required\')\n    data[\'transaction_timestamp\'] = pd.to_datetime(\n        [timestamp(value) for value in data.transaction_timestamp], utc=True).tz_convert(\'Africa/Nairobi\')\n    times = data.transaction_timestamp\n    if (times < start).any() or (times >= target).any():\n        raise ValueError(\'History rows must lie in the attested pre-target interval\')\n    expense = data.loc[data.transaction_type.eq(\'expense\')].copy()\n    if not expense.category.isin(CATEGORIES).all():\n        raise ValueError(\'Unmapped expense category\')\n    dates = pd.date_range(start.tz_localize(None), periods=weeks, freq=\'7D\')\n    panel = pd.DataFrame({\'user_id\': owner, \'week\': dates})\n    for i, category in enumerate(CATEGORIES):\n        subset = expense.loc[expense.category.eq(category)].copy()\n        local = subset.transaction_timestamp.dt.tz_localize(None)\n        subset[\'week\'] = local.dt.to_period(\'W-SUN\').dt.start_time\n        grouped = subset.groupby(\'week\').amount\n        panel[f\'spend_{i}\'] = grouped.sum().reindex(dates, fill_value=0).to_numpy()\n        panel[f\'count_{i}\'] = grouped.size().reindex(dates, fill_value=0).to_numpy()\n    panel[\'total_spend\'] = panel[SPEND_COLS].sum(axis=1)\n    panel[\'total_count\'] = panel[COUNT_COLS].sum(axis=1)\n    return panel\n\n\nclass FrozenCategoryLSTM:\n    def __init__(self, directory, *, trusted=False):\n        if not trusted:\n            raise PermissionError(\'Load only the verified project-owned model/scaler bundle\')\n        directory = Path(directory)\n        for name, expected in HASHES.items():\n            if hashlib.sha256((directory/name).read_bytes()).hexdigest() != expected:\n                raise ValueError(f\'Artifact hash mismatch: {name}\')\n        self.metadata = json.loads((directory/\'metadata.json\').read_text())\n        if (self.metadata[\'categories\'] != CATEGORIES or self.metadata[\'feature_columns\'] != FEATURES\n                or self.metadata[\'sequence_window_weeks\'] != 26):\n            raise ValueError(\'Unsupported category/feature contract\')\n        versions = version_record()\n        for package, expected in {\'tensorflow\': \'2.20.0\', \'keras\': \'3.13.2\', \'scikit-learn\': \'1.6.1\'}.items():\n            if versions[package] != expected:\n                raise ValueError(f\'Expected {package} {expected}, found {versions[package]}\')\n        import tensorflow as tf\n        # Fail rather than use an initialized accelerator supplied by another cell.\n        tf.config.set_visible_devices([], \'GPU\')\n        if tf.config.get_visible_devices(\'GPU\'):\n            raise RuntimeError(\'CPU-only execution required\')\n        with tf.device(\'/CPU:0\'):\n            self.model = tf.keras.models.load_model(directory/\'model.keras\', compile=False, safe_mode=True)\n        if tuple(self.model.input_shape) != (None, 26, 76) or tuple(self.model.output_shape) != (None, 9):\n            raise ValueError(\'Unexpected model dimensions\')\n        with warnings.catch_warnings():\n            warnings.simplefilter(\'error\', InconsistentVersionWarning)\n            self.x_scaler = joblib.load(directory/\'input_scaler.pkl\')\n            self.y_scaler = joblib.load(directory/\'target_scaler.pkl\')\n        for scaler, dimension in ((self.x_scaler, 76), (self.y_scaler, 9)):\n            if type(scaler) is not StandardScaler or scaler.n_features_in_ != dimension:\n                raise ValueError(\'Unexpected scaler type/dimension\')\n            if not np.isfinite(scaler.mean_).all() or not np.isfinite(scaler.scale_).all() or (scaler.scale_ <= 0).any():\n                raise ValueError(\'Invalid frozen scaler statistics\')\n        self.versions = versions\n\n    def feature_tensor(self, panel):\n        features, columns = add_model_features(panel)\n        if columns != FEATURES or len(features) < 26:\n            raise ValueError(\'Invalid feature input\')\n        values = features[FEATURES].tail(26).to_numpy(dtype=np.float32)\n        if not np.isfinite(values).all():\n            raise ValueError(\'Nonfinite feature input\')\n        tensor = self.x_scaler.transform(values).astype(np.float32)[None, ...]\n        if not np.isfinite(tensor).all():\n            raise ValueError(\'Nonfinite scaled feature input\')\n        return tensor\n\n    def predict_tensor(self, tensor):\n        import tensorflow as tf\n        tensor = np.asarray(tensor, dtype=np.float32)\n        if tensor.ndim != 3 or tensor.shape[1:] != (26, 76) or not np.isfinite(tensor).all():\n            raise ValueError(\'Expected finite N x 26 x 76 input\')\n        with tf.device(\'/CPU:0\'):\n            scaled = self.model(tensor, training=False).numpy()\n        with np.errstate(over=\'raise\', invalid=\'raise\'):\n            amounts = np.maximum(0., np.expm1(self.y_scaler.inverse_transform(scaled)))\n        if not np.isfinite(amounts).all():\n            raise ValueError(\'Nonfinite forecast output\')\n        return amounts\n\n    def forecast(self, records, **coverage):\n        panel = history_panel(records, **coverage)\n        amounts = self.predict_tensor(self.feature_tensor(panel))[0]\n        return {\'target_start\': week_boundary(coverage[\'target_start\']).isoformat(),\n                \'category_forecasts_KES\': dict(zip(CATEGORIES, map(float, amounts))),\n                \'total_forecast_KES\': float(amounts.astype(float).sum()),\n                \'history_weeks\': len(panel), \'model\': HASHES[\'model.keras\'],\n                \'quality_status\': \'final_refit_external_evaluation_pending\'}\n', 'check_category_lstm.py': '"""Actual frozen-artifact CPU readiness check on generated histories only."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom category_lstm_frozen import FrozenCategoryLSTM, HASHES, history_panel\n\n\ndef fixture(weeks=80):\n    rows = []\n    start = pd.Timestamp(\'2024-01-01T00:00:00+03:00\')\n    for week in range(weeks):\n        # Deliberate leading/trailing quiet weeks, zero categories and regular bills.\n        if week in (0, weeks-1):\n            continue\n        for day, category, amount in [(1, \'Food\', 700.+week%5*100), (3, \'Transport\', 350.)]:\n            rows.append(dict(user_id=\'synthetic\', transaction_id=f\'{week}-{day}\',\n                transaction_timestamp=(start+pd.Timedelta(weeks=week, days=day, hours=12)).isoformat(),\n                transaction_type=\'expense\', category=category, amount=amount))\n        if week%4 == 0:\n            rows.append(dict(user_id=\'synthetic\', transaction_id=f\'{week}-bill\',\n                transaction_timestamp=(start+pd.Timedelta(weeks=week, days=2, hours=9)).isoformat(),\n                transaction_type=\'expense\', category=\'Rent\', amount=12000.))\n    return pd.DataFrame(rows), dict(owner=\'synthetic\', observed_from=start,\n        target_start=start+pd.Timedelta(weeks=weeks), as_of=start+pd.Timedelta(weeks=weeks, days=2),\n        history_complete=True)\n\n\ndef check(directory):\n    adapter = FrozenCategoryLSTM(directory, trusted=True)\n    tensors = []\n    for weeks in (26, 52, 80):\n        records, coverage = fixture(weeks)\n        panel = history_panel(records, **coverage)\n        assert len(panel) == weeks and panel.total_spend.iloc[-1] == 0\n        tensors.append(adapter.feature_tensor(panel))\n    tensors = np.concatenate(tensors)\n    batch = adapter.predict_tensor(tensors)\n    individual = np.concatenate([adapter.predict_tensor(x[None, ...]) for x in tensors])\n    # CPU batch kernels may differ in rounding; explicit KES tolerance.\n    np.testing.assert_allclose(batch, individual, atol=.02, rtol=1e-5)\n    loaded = FrozenCategoryLSTM(directory, trusted=True)\n    reload = loaded.predict_tensor(tensors)\n    np.testing.assert_array_equal(batch, reload)\n    result = {\'status\': \'passed\', \'scope\': \'generated fixture inference only; no external quality evaluation\',\n              \'CPU_only\': True, \'model_input\': [None,26,76], \'model_output\': [None,9],\n              \'history_lengths_weeks\': [26,52,80], \'forecast_rows\': 3,\n              \'category_outputs\': 27, \'max_batch_single_difference_KES\': float(np.max(abs(batch-individual))),\n              \'max_reload_difference_KES\': float(np.max(abs(batch-reload))),\n              \'artifact_hashes\': HASHES, \'versions\': adapter.versions,\n              \'scaler_fit_rows\': {\'input\': int(adapter.x_scaler.n_samples_seen_),\n                                  \'target\': int(adapter.y_scaler.n_samples_seen_)},\n              \'source_hashes\': {name: hashlib.sha256((Path(__file__).parent/name).read_bytes()).hexdigest()\n                                for name in (\'category_lstm_features.py\',\'category_lstm_frozen.py\')}}\n    return result, batch\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--bundle\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--compare-run\', type=Path)\n    args = p.parse_args()\n    result, batch = check(args.bundle)\n    result[\'synthetic_forecasts_KES\'] = batch.tolist()\n    if args.compare_run:\n        previous = json.loads(args.compare_run.read_text())\n        np.testing.assert_allclose(batch, previous[\'synthetic_forecasts_KES\'], atol=.02, rtol=1e-5)\n        result[\'fresh_process_max_difference_KES\'] = float(np.max(abs(batch-np.array(previous[\'synthetic_forecasts_KES\']))))\n    with args.output.open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(result, f, indent=2, allow_nan=False)\n    print(json.dumps(result, indent=2))\n'}
HASHES = {'category_lstm_features.py': '6e0bf7bb462ffbc3a6443c4116c29a526645b75f19eecac2be48e5511a373631', 'category_lstm_frozen.py': '51f04831677b3570a27b3363696f026debfb0872f52f10e5ba861179ca1864cf', 'check_category_lstm.py': '03f8100ab2fa37be20cfb143737b2550fb9925248034e6c279871cfe4cdf8e65'}
MODULE_DIR = Path('/kaggle/working/lstm_readiness_modules') if Path('/kaggle').exists() else Path('lstm_readiness_modules')
MODULE_DIR.mkdir(exist_ok=True)
if any(name[:-3] in sys.modules for name in SOURCES):
    raise RuntimeError('Restart kernel before rerunning setup to avoid stale imports')
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == HASHES[name]
    path = MODULE_DIR/name
    if path.exists() and path.read_text(encoding='utf-8') != source:
        raise ValueError('Existing source differs')
    with path.open('w', encoding='utf-8', newline='\n') as f:
        f.write(source)
sys.path.insert(0, str(MODULE_DIR.resolve()))
from check_category_lstm import check


In [ ]:
BUNDLE = Path('/kaggle/input/YOUR_PRIVATE_MODEL_DATASET/category_lstm_frozen')
RUN_CHECK = False  # Set True after choosing the bundle path and CPU runtime.
if RUN_CHECK:
    report, forecasts = check(BUNDLE)
    report['synthetic_forecasts_KES'] = forecasts.tolist()
    OUTPUT = Path('/kaggle/working/lstm_readiness.json') if Path('/kaggle').exists() else Path('lstm_readiness.json')
    with OUTPUT.open('x', encoding='utf-8') as f:
        json.dump(report, f, indent=2, allow_nan=False)
    print(json.dumps(report, indent=2))
else:
    print('Configured only. No model loaded or records read.')


## External evaluation boundary

The adapter requires an explicit household, confirmed complete observed history,
Monday-start target and as-of time. At least26 complete weeks are required; retain
all available history for the52-week features. It rejects target-week rows,
unknown expense categories, duplicate IDs, missing values and ambiguous dates.
Income is excluded; confirmed quiet weeks are included.

Wasaa schema mapping remains separate. Evaluate both frozen LSTMs on common
eligible origins if both input contracts can be supported, with a predeclared
comparison. Do not fit scalers, optimize thresholds, or train on external data.
The category bundle is the final all-development refit: its stored development
scores belong to an earlier checkpoint and are not independent scores for this
artifact. The retained total LSTM and finalized Isolation Forest remain unchanged.
